# Mixed-grid aquaplanet

Slab ocean and sea ice on their own displaced-pole grid, coupled to SPEEDY T31L8 through ESMF regridding weights. This run is one command:

```bash
python -m jem.main +configuration=aquaplanet-slab-mixed-grid
```

The ocean and sea ice run on the packaged `DisplacedPoleGrid` SCRIP grid, whose pole sits over land to avoid the polar singularity; the four ESMF weight files mapping it to the atmosphere's T31 grid are all named in `aquaplanet-slab-mixed-grid.yaml`. The notebook below composes the same configuration in Python and calls `jem.runners.run(cfg)` -- the entry point `python -m jem.main` itself uses -- so it can plot what the run wrote.

In [ ]:
from pathlib import Path

from hydra import compose, initialize_config_module

import jem.config  # noqa: F401  -- registers the ${jem_data:}/${jcm_data:} resolvers
from jem import plot, runners

output_dir = (Path("output") / "01-04_mixed_grid_aquaplanet").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## Run it

In [ ]:
with initialize_config_module(config_module="jem.config", version_base="1.3"):
    cfg = compose(config_name="config", overrides=[
        "+configuration=aquaplanet-slab-mixed-grid",
        f"coupled_run.output_dir={output_dir}",
        "coupled_run.subsample=3",       # 10 records out of 30 coupled days
        "coupled_run.checkpoint_path=null",
    ])
result = runners.run(cfg)
result.steps_completed, [p.name for p in result.paths]

## What it wrote

One file per component per chunk, named after the coupled step its chunk starts at (`<component>-<first step>.nc`).

In [ ]:
atm = plot.open_output(output_dir, "atm")
ocn = plot.open_output(output_dir, "ocn")
seaice = plot.open_output(output_dir, "seaice")
list(ocn.data_vars)

## Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# The atmosphere's field has 1-D lat/lon (its own T31 grid); the
# ocean's has 2-D lat/lon over the displaced-pole grid's own index
# dimensions -- map_plot draws each on its own grid, unregridded.
humidity = atm["specific_humidity"].sel(level=1.0, method="nearest").isel(time=-1)
plot.map_plot(humidity, ax=axes[0],
              title="Atmosphere: surface specific humidity [kg/kg] (T31 grid)")

sst = ocn["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst, ax=axes[1],
              title="Ocean: sea surface temperature [°C] (displaced-pole grid)")
plt.tight_layout()